# Detalhes do projeto: 

- Versão Python: 3.14.7

# 2. DESAFIO
Você acaba de ser contratado como Analista de Dados Júnior em uma empresa de varejo que possui um histórico de vendas em formato CSV e precisa de um relatório analítico para a reunião trimestral da diretoria. O time de negócios quer entender:

- Como as vendas se comportam ao longo do tempo (por mês e por trimestre);
- Quais produtos e categorias geram mais receita?
- Quais regiões têm melhor desempenho?
- Quais clientes são mais valiosos (segmentação simples por faixa de gasto)?
- Quantas vendas individuais tiveram receita acima da média geral de receita por venda?


In [1]:
# Agrupando importações necessárias para o projeto 
import os
import csv
import json
import random
import re
from datetime import datetime, timedelta
from funcoes_dataset import carregar_dataset, inspecionar_dados, limpar_dados, converter_data

# RF01 – Criar ou Carregar o Dataset de Vendas

Gerando sementes do dataset conforme recomendação do guia para o projeto. (Desenvolvedor(a) em IA para Análise Preditiva [T4])

Dei preferência por modularizar o código, já que não faz parte do escopo principal e não manter o código completo no mesmo arquivo do projeto principal.

In [2]:
# Importando função para gerar as sementes do dataset
from gerar_sementes import gerar_dataset_vendas

# chamando a função
gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42)

Dataset gerado com 200 registros em vendas.csv.


# RF02 – Inspecionar e Descrever os Dados

Observações: O arquivo, ou pelo menos o nome do arquivo e formato precisam existir antes de executar o comando.

Em versões futuras, criar uma função para validar a existência do arquivo antes de importar um arquivo inexistente e informar na tela para o usuário que o arquivo não existe antes de "quebrar" fluxo do código.

Importando o dataset

In [3]:
# carregar_dataset(caminho_csv="vendas.csv")

registros = carregar_dataset("vendas.csv")
inspecionar_dados(registros)


=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': '1', 'preco_unitario': '1939.76'}
{'id_venda': '4', 'data_venda': '2025-11-06', 'cliente': 'Cliente_017', 'p

[{'id_venda': '1',
  'data_venda': '2025-05-21',
  'cliente': 'cliente#016',
  'produto': 'Mouse',
  'categoria': 'Perifericos',
  'regiao': 'Sudeste',
  'quantidade': '2',
  'preco_unitario': '102.9'},
 {'id_venda': '2',
  'data_venda': '2025-09-16',
  'cliente': 'Cliente_039',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '',
  'preco_unitario': '3204.57'},
 {'id_venda': '3',
  'data_venda': '2025-03-23',
  'cliente': 'Cliente_045',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Centro-Oeste',
  'quantidade': '1',
  'preco_unitario': '1939.76'},
 {'id_venda': '4',
  'data_venda': '2025-11-06',
  'cliente': 'Cliente_017',
  'produto': 'Notebook',
  'categoria': 'Computadores',
  'regiao': 'Norte',
  'quantidade': '6',
  'preco_unitario': '3864.87'},
 {'id_venda': '5',
  'data_venda': '2025-07-05',
  'cliente': 'Cliente_037',
  'produto': 'Tablet',
  'categoria': 'Celulares',
  'regiao': 'Sul',
  'quantidade': '10',
  'pre

# RF03 – Limpar e Tratar os Dados (datetime e regex)

Implemente uma função de limpeza que trate, no mínimo:
- Espaços extras em campos de texto, usando .strip();
- Datas inválidas: converter data_venda com datetime.strptime(...) dentro de um try/except e remover os registros que não puderam ser convertidos;
- Valores ausentes nos campos críticos quantidade e preco_unitario, descartando o registro quando algum deles estiver vazio;
- Tipos de dados: garantir int para quantidade e float para preco_unitario;
Ruído nos nomes de cliente, usando o módulo re: remover caracteres não alfanuméricos, padronizar o formato para - Cliente_NNN e sinalizar os registros fora do padrão;
- Relatório de limpeza: exibir no console quantos registros entraram, quantos foram removidos (por motivo) e quantos permaneceram.

In [4]:
limpar_dados(registros)


=== RELATORIO DE LIMPEZA ===
{'iniciais': 200, 'removidos_data': 4, 'removidos_nulos': 13, 'finais': 183}


([{'id_venda': '1',
   'data_venda': datetime.datetime(2025, 5, 21, 0, 0),
   'cliente': 'cliente016',
   'produto': 'Mouse',
   'categoria': 'Perifericos',
   'regiao': 'Sudeste',
   'quantidade': 2,
   'preco_unitario': 102.9,
   'cliente_fora_do_padrao': True},
  {'id_venda': '3',
   'data_venda': datetime.datetime(2025, 3, 23, 0, 0),
   'cliente': 'Cliente_045',
   'produto': 'Tablet',
   'categoria': 'Celulares',
   'regiao': 'Centro-Oeste',
   'quantidade': 1,
   'preco_unitario': 1939.76,
   'cliente_fora_do_padrao': False},
  {'id_venda': '4',
   'data_venda': datetime.datetime(2025, 11, 6, 0, 0),
   'cliente': 'Cliente_017',
   'produto': 'Notebook',
   'categoria': 'Computadores',
   'regiao': 'Norte',
   'quantidade': 6,
   'preco_unitario': 3864.87,
   'cliente_fora_do_padrao': False},
  {'id_venda': '5',
   'data_venda': datetime.datetime(2025, 7, 5, 0, 0),
   'cliente': 'Cliente_037',
   'produto': 'Tablet',
   'categoria': 'Celulares',
   'regiao': 'Sul',
   'quantidade'

In [13]:
#converter_data(registros)

print(registros[1])

{'id_venda': '2', 'data_venda': datetime.datetime(2025, 9, 16, 0, 0), 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}


In [ ]:
# # conversao de data com tratamento de erro
# try:
#     data = datetime.strptime(texto_data, "%Y-%m-%d")
# except ValueError:
#     data = None                       # marca como invalida

# # limpeza de texto com expressao regular
# nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", nome.strip())

# # validacao de formato com padrao compilado
# padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

NameError: name 'texto_data' is not defined